In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import wandb

In [ ]:
DATASET = 'CIFAR100'  # Options: 'CIFAR10', 'CIFAR100', 'MNIST', 'FashionMNIST'


In [ ]:
import wandb
wandb.login() # Assumes WANDB_API_KEY is set in environment or user logs in manually

# Configuration for SPLIT_STRATEGY will be in the 'train-model' cell
# NUM_SUBSETS will also be determined there based on the strategy.
wandb.init(project=f'{DATASET.lower()}-cnn-subsets-pytorch', name='cifar100-cnn-subsets-run', config={
    'learning_rate': 0.001, # optimizer's lr
    'subset_epochs': 3, # SUBSET_NUM_EPOCHS from 'train-model' cell
    'batch_size': 64, # trainloader batch_size (default for subsets)
    'test_batch_size': 1000, # testloader batch_size
    'dataset': DATASET
})


In [ ]:
# Prepare dataset and data loaders based on selected DATASET
if DATASET == 'CIFAR100':
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5071,0.4867,0.4408], std=[0.2675,0.2565,0.2761])
    ])
    full_trainset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
    testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
    input_channels = 3
    image_size = 32
    num_output_classes = 100
elif DATASET == 'CIFAR10':
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914,0.4822,0.4465), (0.247,0.243,0.261))
    ])
    full_trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    input_channels = 3
    image_size = 32
    num_output_classes = 10
elif DATASET == 'MNIST':
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    full_trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    input_channels = 1
    image_size = 28
    num_output_classes = 10
elif DATASET == 'FashionMNIST':
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.2860,), (0.3530,))
    ])
    full_trainset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
    testset = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
    input_channels = 1
    image_size = 28
    num_output_classes = 10
else:
    raise ValueError(f'Unsupported dataset {DATASET}')

testloader = torch.utils.data.DataLoader(testset, batch_size=1000, shuffle=False)

def split_train_data(full_trainset, split_by='random', num_subsets_random=10):
    subsets = []
    if split_by == 'class':
        num_classes = len(set(full_trainset.targets))
        class_indices = [[] for _ in range(num_classes)]
        for i, label in enumerate(full_trainset.targets):
            class_indices[label].append(i)
        for idxs in class_indices:
            if idxs:
                subsets.append(torch.utils.data.Subset(full_trainset, idxs))
        print(f'Data split into {len(subsets)} subsets by class.')
    elif split_by == 'random':
        length = len(full_trainset)//num_subsets_random
        subsets = [torch.utils.data.Subset(full_trainset, range(i*length, (i+1)*length)) for i in range(num_subsets_random)]
        print(f'Data split into {len(subsets)} random subsets.')
    else:
        raise ValueError('Unknown split strategy')
    return subsets


In [ ]:
# Define the CNN architecture
class Net(nn.Module):
    def __init__(self, num_classes, input_channels, image_size):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=3, stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        fc_input_dim = 64 * (image_size // 4) * (image_size // 4)
        self.fc1 = nn.Linear(fc_input_dim, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Instantiate the model
model = Net(num_classes=num_output_classes, input_channels=input_channels, image_size=image_size)

print(model)


In [ ]:
# Define the loss function
criterion = nn.CrossEntropyLoss()

# Define the optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Configuration for data splitting
SPLIT_STRATEGY = "class"  # Options: "class", "superclass", "random"
NUM_SUBSETS_RANDOM = 10 # Used only if SPLIT_STRATEGY is "random"
SUBSET_NUM_EPOCHS = 3 # Number of epochs to train on each subset

# Determine NUM_SUBSETS based on strategy
if SPLIT_STRATEGY == "class":
    NUM_SUBSETS = 100
elif SPLIT_STRATEGY == "superclass":
    NUM_SUBSETS = 20
else: # random
    NUM_SUBSETS = NUM_SUBSETS_RANDOM

# Log strategy and num_subsets to wandb if run is active
if wandb.run:
    wandb.config.update({"SPLIT_STRATEGY": SPLIT_STRATEGY, "NUM_SUBSETS": NUM_SUBSETS}, allow_val_change=True)

# Prepare the subsets
train_subsets = split_train_data(full_trainset, split_by=SPLIT_STRATEGY, num_subsets_random=NUM_SUBSETS_RANDOM)

overall_val_accuracies_after_subset_training = []

print(f"Starting training process with SPLIT_STRATEGY='{SPLIT_STRATEGY}' on {len(train_subsets)} actual subsets...")

# Define CIFAR-100 unnormalization parameters for plotting samples
cifar_mean_unnorm = torch.tensor([0.5071, 0.4867, 0.4408]).view(3, 1, 1)
cifar_std_unnorm = torch.tensor([0.2675, 0.2565, 0.2761]).view(3, 1, 1)

for subset_idx, current_subset_data in enumerate(train_subsets):
    print(f"--- Training on Subset {subset_idx + 1}/{NUM_SUBSETS} ---")
    
    if len(current_subset_data) == 0:
        print(f"Subset {subset_idx + 1} is empty. Skipping.")
        # Append a placeholder for accuracy if needed, e.g., 0 or None, or handle in plotting
        overall_val_accuracies_after_subset_training.append(0) # Or None
        continue

    current_trainloader = torch.utils.data.DataLoader(current_subset_data, batch_size=64, shuffle=True)
    
    subset_train_losses = [] # To store losses for epochs within this subset

    for epoch in range(SUBSET_NUM_EPOCHS):
        model.train() # Set model to training mode
        running_loss = 0.0
        for i, data in enumerate(current_trainloader, 0):
            images, labels = data
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        epoch_loss = running_loss / len(current_trainloader)
        subset_train_losses.append(epoch_loss)
        print(f'Subset {subset_idx + 1}, Epoch [{epoch + 1}/{SUBSET_NUM_EPOCHS}], Training Loss: {epoch_loss:.4f}')
        
        # Log metrics to W&B per epoch within subset
        if wandb.run:
            wandb.log({
                "subset_epoch": epoch + 1,
                f"subset_{subset_idx+1}_train_loss": epoch_loss,
                "global_step": subset_idx * SUBSET_NUM_EPOCHS + epoch
            })

    # Report training loss for the subset (last epoch's loss)
    last_epoch_loss = subset_train_losses[-1] if subset_train_losses else float('nan')
    print(f'--- Finished training on Subset {subset_idx + 1} ---')
    print(f'Last Training Loss for Subset {subset_idx + 1}: {last_epoch_loss:.4f}')
    
    # Validation on the full test set
    model.eval() # Set model to evaluation mode
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader: # testloader uses the full test set
            images, labels = data
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    current_val_accuracy = 100 * correct / total
    overall_val_accuracies_after_subset_training.append(current_val_accuracy)
    print(f'Validation Accuracy on Full Test Set after Subset {subset_idx + 1}: {current_val_accuracy:.2f} %')
    
    # Log validation accuracy to W&B after each subset training
    if wandb.run:
        wandb.log({
            f"val_accuracy_after_subset_{subset_idx+1}": current_val_accuracy,
            "subset_trained": subset_idx + 1
        })
    
    # Display a few samples from the current training subset
    print(f'Displaying samples from Subset {subset_idx + 1}:')
    model.eval() # Keep model in eval mode for consistency
    num_samples_to_show = 3
    
    # Ensure current_subset_data is not empty before trying to get samples
    if len(current_subset_data) > 0:
        # Get a few random samples from the current_subset_data
        sample_indices = np.random.choice(len(current_subset_data), min(num_samples_to_show, len(current_subset_data)), replace=False)
        
        fig, axes = plt.subplots(1, len(sample_indices), figsize=(len(sample_indices) * 3, 3))
        if len(sample_indices) == 1: # if only one sample, axes might not be an array
            axes = [axes] 

        for i, data_idx in enumerate(sample_indices):
            image, label = current_subset_data[data_idx] # image is a Tensor (C,H,W)
            
            # Unnormalize for display
            unnormalized_image = image.cpu() * cifar_std_unnorm + cifar_mean_unnorm
            unnormalized_image_np = unnormalized_image.permute(1, 2, 0).numpy() # H, W, C
                                                                     
            ax = axes[i]
            ax.imshow(unnormalized_image_np)
            ax.set_title(f"True: {label}")
            ax.axis('off')
            
            # Optional: Add prediction
            with torch.no_grad():
                image_tensor = image.unsqueeze(0)
                output = model(image_tensor)
                _, predicted = torch.max(output.data, 1)
                ax.set_xlabel(f"Pred: {predicted.item()}")

        plt.tight_layout()
        plt.show()
        
        # Log image samples to W&B
        if wandb.run:
            log_images = []
            for data_idx in sample_indices:
                img_tensor, true_label = current_subset_data[data_idx]
                # Model prediction for caption
                with torch.no_grad():
                    pred_label = model(img_tensor.unsqueeze(0).to(next(model.parameters()).device)).data.max(1)[1].item()
                log_images.append(wandb.Image(img_tensor, caption=f"True: {true_label}, Pred: {pred_label}"))
            
            wandb.log({
                f"subset_{subset_idx+1}_sample_images": log_images
            }, commit=True)
            
    else:
        print("No samples to display from this subset as it was empty.")
    print(f"--- End of report for Subset {subset_idx + 1} ---\n")

print('Finished Training on all subsets')
# The variable `overall_val_accuracies_after_subset_training` can be used later for plotting

In [ ]:
# Final evaluation of the model
model.eval() # Set model to evaluation mode
correct = 0
total = 0
with torch.no_grad(): # Disable gradient calculations
    for data in testloader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

final_accuracy = 100 * correct / total
print(f'Final accuracy of the network on the {len(testset)} test images: {final_accuracy:.2f} %')

# Log final accuracy to W&B
if wandb.run:
    wandb.log({"final_test_accuracy": final_accuracy})

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Ensure model is in evaluation mode
model.eval()

# Get a few random samples from the testset
num_samples = 5
# Create a random permutation of indices from the testset
random_indices = np.random.choice(len(testset), num_samples, replace=False)

# Create subplots: num_samples rows, 2 columns (original, correct_example_if_misclassified)
if num_samples == 1:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5)) # Adjust figsize as needed
    axes = np.array([axes]) # Make it a 2D array for consistent indexing
else:
    fig, axes = plt.subplots(num_samples, 2, figsize=(10, 3 * num_samples))

cifar_mean_for_unnorm = torch.tensor([0.5071, 0.4867, 0.4408]).view(3, 1, 1)
cifar_std_for_unnorm = torch.tensor([0.2675, 0.2565, 0.2761]).view(3, 1, 1)

for i, idx in enumerate(random_indices):
    image, label = testset[idx]
    
    # Unnormalize the original image
    unnormalized_image = image.cpu() * cifar_std_for_unnorm + cifar_mean_for_unnorm
    display_image = unnormalized_image.permute(1, 2, 0).numpy()
                                                             
    # Plot the original image on the left subplot
    ax_orig = axes[i, 0]
    ax_orig.imshow(display_image)
    ax_orig.set_title(f"True: {label}")
    ax_orig.axis('off')

    # Make a prediction
    # We need to add a batch dimension (B) and channel dimension (C) to the image (H, W) -> (B, C, H, W)
    # and move it to the same device as the model if necessary.
    with torch.no_grad():
        # Assuming image is (C, H, W) and model expects (B, C, H, W)
        # Also, ensure the image tensor is on the same device as the model
        image_tensor = image.unsqueeze(0) 
        # If your model is on a GPU, move the tensor to GPU:
        # if next(model.parameters()).is_cuda:
        #    image_tensor = image_tensor.to('cuda')
            
        output = model(image_tensor)
        _, predicted = torch.max(output.data, 1)
        predicted_item = predicted.item()
        ax_orig.set_xlabel(f"Pred: {predicted_item}")

    # Handle the right subplot (for correct class example or turn off)
    ax_example = axes[i, 1]
    if predicted_item != label:
        ax_orig.set_title(f"True: {label}, Pred: {predicted_item} (Misclassified!)")
        # Find an example of the MISPREDICTED class
        found_mispredicted_example = False
        for example_idx in range(len(testset)):
            if example_idx == idx: # Don't pick the same image
                continue
            example_image_candidate, example_label_candidate = testset[example_idx]
            if example_label_candidate == predicted_item: # Found an example of what the model predicted
                unnormalized_example_image = example_image_candidate.cpu() * cifar_std_for_unnorm + cifar_mean_for_unnorm
                display_example_image = unnormalized_example_image.permute(1, 2, 0).numpy()
                ax_example.imshow(display_example_image)
                ax_example.set_title(f"Example of Predicted: {predicted_item}")
                ax_example.axis('off')
                found_mispredicted_example = True
                break
        if not found_mispredicted_example:
            ax_example.set_title(f"No example for Predicted: {predicted_item} found")
            ax_example.axis('off')
    else:
        ax_example.axis('off') # Correctly classified, turn off the right subplot

plt.tight_layout()
plt.show()

# Print predicted and true labels below the images for clarity
print("Details of the samples shown above:")
for i, idx in enumerate(random_indices):
    _, label = testset[idx] # Original label
    
    # Re-predict to match the loop structure for printing
    image_tensor = testset[idx][0].unsqueeze(0)
    # if next(model.parameters()).is_cuda:
    #    image_tensor = image_tensor.to('cuda')
        
    with torch.no_grad():
        output = model(image_tensor)
        _, predicted = torch.max(output.data, 1)
    
    print(f"Sample {i+1}: True Label = {label}, Predicted Label = {predicted.item()}")


In [ ]:
import matplotlib.pyplot as plt

# Ensure overall_val_accuracies_after_subset_training is available from the training cell
# Ensure NUM_SUBSETS is available (or derive from the length of the accuracy list)

if 'overall_val_accuracies_after_subset_training' in globals() and \
   len(overall_val_accuracies_after_subset_training) > 0:
    
    subset_numbers = range(1, len(overall_val_accuracies_after_subset_training) + 1)

    plt.figure(figsize=(10, 6))
    plt.plot(subset_numbers, overall_val_accuracies_after_subset_training, marker='o', linestyle='-')
    plt.title('Validation Accuracy on Full Test Set vs. Training Subset')
    plt.xlabel('Number of Subsets Trained On')
    plt.ylabel('Validation Accuracy (%)')
    plt.xticks(subset_numbers) # Ensure all subset numbers are ticked
    plt.grid(True)
    plt.show()
    
    print("\nSummary of Validation Accuracies per Subset:")
    for i, acc in enumerate(overall_val_accuracies_after_subset_training):
        print(f"After training on Subset {i+1}: {acc:.2f}%")
else:
    print("Training data for plotting not available. Please run the 'train-model' cell first.")

In [ ]:
# Ensure W&B run is properly closed
if wandb.run:
    wandb.finish()